In [1]:
from langchain_core.documents import Document

In [ ]:
# Creating a document
doc=Document(
    page_content="Ghar ki yaad nhi aai tujhe jassi?",
    metadata = {
        "source": "dhurandhar.txt",
        "pages": 1,
        "author":"Aditya Dhar",
        "date_created":"2026-03-18"
    }
)
doc

Document(metadata={'source': 'dhurandhar.txt', 'pages': 1, 'author': 'Aditya Dhar', 'date_created': '2025-01-01'}, page_content='Ghar ki yaad nhi aai tujhe jassi?')

In [12]:
# Directory loader: Loads all the pdf files in a specific directory

from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
dir_loader =DirectoryLoader(
    "../data/pdfs",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
documents = dir_loader.load()
documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-03-05T09:43:57+01:00', 'source': '../data/pdfs/EJ1172284.pdf', 'file_path': '../data/pdfs/EJ1172284.pdf', 'total_pages': 11, 'format': 'PDF 1.6', 'title': '', 'author': 'agimeno', 'subject': '', 'keywords': '', 'moddate': '2018-03-12T10:24:10-04:00', 'trapped': '', 'modDate': "D:20180312102410-04'00'", 'creationDate': "D:20180305094357+01'00'", 'page': 0}, page_content="The EUROCALL Review, Volume 25, No. 2, September 2017 \n \n18 \nResearch paper \n \nA look at advanced learners’ use of mobile devices for \nEnglish language study: Insights from interview data \nMariusz Kruk \nUniversity of Zielona Gora, Poland \n______________________________________________________________ \nmkruk @ uz.zgora.pl \n  \nAbstract \nThe paper discusses the results of a study which explored advanced learners of English \nengagement with their mobile devices to develop learning experiences that m

In [14]:
# Chunking Data

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks=split_documents(documents)

Split 22 documents into 126 chunks

Example chunk:
Content: The EUROCALL Review, Volume 25, No. 2, September 2017 
 
18 
Research paper 
 
A look at advanced learners’ use of mobile devices for 
English language study: Insights from interview data 
Mariusz Kru...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-03-05T09:43:57+01:00', 'source': '../data/pdfs/EJ1172284.pdf', 'file_path': '../data/pdfs/EJ1172284.pdf', 'total_pages': 11, 'format': 'PDF 1.6', 'title': '', 'author': 'agimeno', 'subject': '', 'keywords': '', 'moddate': '2018-03-12T10:24:10-04:00', 'trapped': '', 'modDate': "D:20180312102410-04'00'", 'creationDate': "D:20180305094357+01'00'", 'page': 0}


In [4]:
# Embedding
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
import os
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18175.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [6]:
# Vector Store
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [17]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 126 texts...


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.24it/s]

Generated embeddings with shape: (126, 384)
Adding 126 documents to vector store...
Successfully added 126 documents to vector store
Total documents in collection: 126


In [18]:
# Retrive pipeline from vector store
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [19]:
rag_retriever.retrieve(query="Tell me about Research Philosophy")

Retrieving documents for query: 'Tell me about Research Philosophy'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.84s/it]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_89f9106e_81',
  'content': 'associated with the design aspect of the project: \n “Research Topic”  Methodology \n “Research Topic”  “Research Methods” \n “Research Topic”  “Project Design” \n “Research Topic”  Models \n \n2.1. Research Philosophy \nThe degree to which we specific our research approach varies \nfrom research field to research field; some fields make implicit \nassumptions about the type of research being undertaken \nwhereas others prefer if the research approach is fully \narticulated. We will know how much of this content is required \nby looking at other similar papers in the domain. With this in \nmind we will discuss some of the common questions to reflect \non to explore our approach. \n \nResearch Paradigm \nThe Research Paradigm is concerned with our beliefs and \nassumptions about research and science. The most common \nparadigms are presented below: \n• \nPositivism: Does our experiment assume that we can \nobtain completely accurate results, and 

In [20]:
rag_retriever.retrieve(query="Tell me about Usage of Mobile Devices")

Retrieving documents for query: 'Tell me about Usage of Mobile Devices'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3966ff46_24',
  'content': 'mobile devices, reasons for using mobile devices, resources and \ntools, mobile \nencounters, language practiced and study performance. \n4.1. Usage of mobile devices \nTable 1 shows the study participants’ mobile devices (MobDs) usage descriptions. The \ntable demonstrates that smartphones were the most often used mobile devices by the \nstudents. In addition, the numerical information in the table indicates that the \nparticipants, on average, had been using them for English language study for about \n3.80 years (minimum 2, maximum 6 years). 9 (45%) and 11 (55%) of the subjects \nstarted using their mobile devices at senior high school and university, respectively. It \nshould also be added that, with the exception of one student (i.e. S9), all the other \nparticipants claimed to use their mobile devices in order to learn English much more \nfrequently with time. Finally, more than half of the students (55%) regarded themselves',
  'metadata':